# <span style='color:blue'>PART 3: WEB SCRAPING & HTML PARSING</span> ⏱️ 

---

## <span style='color:blue'>3.1 HTML Structure: Building Blocks of Web Pages</span>

**HTML is like a tree:**

```html
<html>          ← Root
  <head>        ← Branch
    <title>Page</title>   ← Leaf
  </head>
  <body>        ← Branch
    <div class="container">   ← Branch
      <h1>Title</h1>    ← Leaf
      <p>Content</p>    ← Leaf
    </div>
  </body>
</html>
```

**Key Concepts:**

```html
<!-- Element: <tag>content</tag> -->
<h1>This is a heading</h1>

<!-- Attributes: properties of elements -->
<div class="container" id="main" data-value="123">

<!-- Classes: for styling (can have multiple) -->
<div class="card featured important">

<!-- IDs: unique identifier (only one per page) -->
<div id="header">

<!-- Links -->
<a href="https://example.com">Click me</a>

<!-- Images -->
<img src="image.jpg" alt="Description">

<!-- Lists -->
<ul>
  <li>Item 1</li>
  <li>Item 2</li>
</ul>
```

## <span style='color:blue'>3.2 CSS Selectors: Finding Elements</span>

**CSS selectors** are patterns used to find and target specific HTML elements. They are the primary way to navigate and extract data from parsed HTML.

In [1]:
from bs4 import BeautifulSoup

# Sample HTML structure used throughout this section
html = """
<div class="container">
  <article class="post" id="post-1">
    <h2 class="title">First Post</h2>
    <p class="content">This is content</p>
    <span class="author" data-id="123">Ahmed</span>
    <div class="tags">
      <a href="/tag/python">Python</a>
      <a href="/tag/data">Data</a>
    </div>
  </article>

  <article class="post featured" id="post-2">
    <h2 class="title">Second Post</h2>
    <p class="content">More content</p>
    <span class="author" data-id="456">Sara</span>
  </article>
</div>
"""

soup = BeautifulSoup(html, 'lxml')

# 1. By tag name — selects ALL <h2> elements
titles = soup.select('h2')
print("All titles:", [t.text for t in titles])

# 2. By class — selects elements with class='post'
posts = soup.select('.post')
print(f"Found {len(posts)} posts")

# 3. By ID — selects the single element with id='post-1'
post1 = soup.select_one('#post-1')
print("Post 1 title:", post1.select_one('h2').text)

# 4. Descendant (space = any level deep) — finds .author inside .post
authors = soup.select('.post .author')
print("Authors:", [a.text for a in authors])

# 5. Direct child (> = immediate child only)
container_children = soup.select('.container > article')
print(f"Direct children: {len(container_children)}")

# 6. Multiple classes — selects elements that have BOTH classes
featured = soup.select('.post.featured')
print(f"Featured posts: {len(featured)}")

# 7. Attribute selectors — [attr*='val'] means 'contains'
python_links = soup.select('a[href*="python"]')
print("Python links:", [l['href'] for l in python_links])

# 8. Starts with — [attr^='val'] means 'starts with'
tag_links = soup.select('a[href^="/tag"]')
print("Tag links:", len(tag_links))

# 9. Combine selectors — chain tag + class + descendant
post_titles = soup.select('article.post h2.title')
print("Post titles:", [t.text for t in post_titles])

# 10. Nth child — positional selectors
first_post = soup.select_one('article:first-child')
last_post = soup.select_one('article:last-child')
second_post = soup.select_one('article:nth-child(2)')

All titles: ['First Post', 'Second Post']
Found 2 posts
Post 1 title: First Post
Authors: ['Ahmed', 'Sara']
Direct children: 2
Featured posts: 1
Python links: ['/tag/python']
Tag links: 2
Post titles: ['First Post', 'Second Post']


### <span style='color:red'>📌 Selector Cheat Sheet</span>

| **Selector** | **Example** | **Meaning** |
|---|---|---|
| `tag` | `div` | All `<div>` elements |
| `.class` | `.post` | Elements with `class="post"` |
| `#id` | `#main` | Element with `id="main"` |
| `tag.class` | `div.post` | `<div>` with `class="post"` |
| `tag tag` | `div p` | `<p>` inside `<div>` (any level) |
| `tag > tag` | `div > p` | Direct `<p>` child of `<div>` |
| `[attr]` | `[href]` | Elements with `href` attribute |
| `[attr="val"]` | `[href="/"]` | Exact match |
| `[attr^="val"]` | `[href^="http"]` | Starts with |
| `[attr$="val"]` | `[src$=".jpg"]` | Ends with |
| `[attr*="val"]` | `[class*="post"]` | Contains |
| `:first-child` | `li:first-child` | First child |
| `:last-child` | `li:last-child` | Last child |
| `:nth-child(n)` | `li:nth-child(2)` | Second child |

## <span style='color:blue'>3.3 BeautifulSoup Methods</span>

Three main methods for finding elements:
- **`find()`** — returns the **first** matching element
- **`find_all()`** — returns **all** matching elements as a list
- **`select()`** / **`select_one()`** — uses CSS selectors (**recommended**)

In [2]:
# Reuse the soup object from the previous cell

# Method 1: find() - returns the FIRST match only
first_post = soup.find('article', class_='post')
print(first_post.find('h2').text)

# Method 2: find_all() - returns ALL matches as a list
all_posts = soup.find_all('article', class_='post')
for post in all_posts:
    print(post.find('h2').text)

# Method 3: select() - CSS selectors (RECOMMENDED for complex queries)
posts = soup.select('article.post')

# Extracting data from each matched element
for post in posts:
    # Get text content (strip=True removes leading/trailing whitespace)
    title = post.select_one('h2').get_text(strip=True)

    # Get attribute value using .get() (safer than direct dict access)
    post_id = post.get('id')  # same as post['id'] but doesn't raise KeyError
    author_id = post.select_one('.author').get('data-id')  # custom data-* attribute

    # Get text of all matching child elements as a list
    tags = [link.text for link in post.select('.tags a')]

    print(f"ID: {post_id}")
    print(f"Title: {title}")
    print(f"Author ID: {author_id}")
    print(f"Tags: {tags}")
    print("-" * 50)

# Navigate the tree using relationship properties
element = soup.select_one('h2')
parent = element.parent  # immediate parent element
next_sibling = element.find_next_sibling()  # next sibling element
previous = element.find_previous('div')  # previous <div> anywhere above

# Handle missing elements SAFELY — always check before accessing
price = soup.select_one('.price')
price_text = price.get_text() if price else 'N/A'  # fallback if element not found

# Use .get() with a default value for missing attributes
link = soup.select_one('a')
href = link.get('href', '#') if link else '#'  # default '#' if no href

First Post
First Post
Second Post
ID: post-1
Title: First Post
Author ID: 123
Tags: ['Python', 'Data']
--------------------------------------------------
ID: post-2
Title: Second Post
Author ID: 456
Tags: []
--------------------------------------------------


## <span style='color:blue'>3.4 Real Example: Scraping Book Data</span>

**Scenario**: Scrape book information from [Books to Scrape](http://books.toscrape.com/) — a safe practice site designed for learning web scraping.

This section builds a full **`BookScraper`** class that:
- Scrapes multiple pages with **pagination**
- Follows links to get **detailed book info**
- Converts data to a **pandas DataFrame**
- Performs **analysis** on the collected data

In [3]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time
from urllib.parse import urljoin  # Safely combines base URL + relative link


class BookScraper:
    """
    Scrape book data from books.toscrape.com
    """

    def __init__(self):
        self.base_url = 'http://books.toscrape.com'
        # Use a Session for persistent headers and connection reuse (more efficient)
        self.session = requests.Session()
        self.session.headers.update(
            {'User-Agent': 'Mozilla/5.0 (Educational Purpose) BookScraper/1.0'}
        )

    def scrape_page(self, url):
        """
        Scrape books from a single page.

        Returns:
            tuple: (list of books, next page URL)
        """
        try:
            response = self.session.get(url, timeout=10)
            response.raise_for_status()  # Raises exception for HTTP errors (4xx, 5xx)

            soup = BeautifulSoup(response.text, 'lxml')
            books = []

            # Find all book articles on the page
            for article in soup.select('article.product_pod'):
                book = {}

                # Title — stored in the 'title' attribute of the <a> tag (not text)
                title_element = article.select_one('h3 a')
                book['title'] = title_element.get('title')

                # Price — remove '£' symbol and convert to float
                price_text = article.select_one('.price_color').text
                book['price'] = float(price_text.replace('Â£', ''))

                # Availability — check if 'In stock' is in the text
                availability = article.select_one('.availability').text.strip()
                book['in_stock'] = 'In stock' in availability

                # Rating — stored as a CSS class like "star-rating Three"
                # We grab the second class name and map it to a number
                rating_element = article.select_one('.star-rating')
                rating_text = rating_element.get('class')[1]  # e.g., 'Three'
                rating_map = {'One': 1, 'Two': 2, 'Three': 3, 'Four': 4, 'Five': 5}
                book['rating'] = rating_map.get(rating_text, 0)

                # Book URL — combine base URL with relative link
                book_link = article.select_one('h3 a')['href']
                book['url'] = urljoin(url, book_link)

                books.append(book)

            # Find the 'next' button for pagination
            next_link = soup.select_one('.next a')
            next_url = urljoin(url, next_link['href']) if next_link else None

            return books, next_url

        except Exception as e:
            print(f"Error scraping {url}: {e}")
            return [], None

    def scrape_book_details(self, book_url):
        """
        Scrape detailed information from a book's individual page.
        """
        try:
            response = self.session.get(book_url, timeout=10)
            soup = BeautifulSoup(response.text, 'lxml')

            details = {}

            # Description — sibling <p> after the #product_description header
            desc_element = soup.select_one('#product_description ~ p')
            details['description'] = (
                desc_element.text if desc_element else 'No description'
            )

            # Category — extracted from the breadcrumb navigation
            breadcrumb = soup.select('.breadcrumb li')
            details['category'] = (
                breadcrumb[2].text.strip() if len(breadcrumb) > 2 else 'Unknown'
            )

            # Product information table — extract UPC and review count
            info_table = soup.select('.table.table-striped tr')
            for row in info_table:
                header = row.select_one('th').text.strip()
                value = row.select_one('td').text.strip()
                if header == 'UPC':
                    details['upc'] = value
                elif header == 'Number of reviews':
                    details['num_reviews'] = int(value)

            return details

        except Exception as e:
            print(f"Error fetching details: {e}")
            return {}

    def scrape_all_pages(self, max_pages=None):
        """
        Scrape multiple pages of books using pagination.
        """
        all_books = []
        url = f"{self.base_url}/catalogue/page-1.html"
        page = 1

        # Loop continues while there's a next page AND we haven't hit max_pages
        while url and (max_pages is None or page <= max_pages):
            print(f"📖 Scraping page {page}...")
            books, next_url = self.scrape_page(url)

            if books:
                all_books.extend(books)
                print(f"  Found {len(books)} books")

            url = next_url
            page += 1

            # Be polite — wait 1 second between requests to avoid overloading the server
            if url:
                time.sleep(1)

        print(f"\n✓ Total books scraped: {len(all_books)}")
        return all_books

    def scrape_with_details(self, max_pages=2, max_details=10):
        """
        Scrape books and fetch detailed info for first N books.
        """
        # Step 1: Get basic info for all books across pages
        books = self.scrape_all_pages(max_pages=max_pages)

        # Step 2: Fetch details for first N books only (to limit requests)
        print(f"\n📚 Fetching details for {max_details} books...")
        for i, book in enumerate(books[:max_details]):
            print(f"  {i+1}/{max_details}: {book['title'][:50]}...")
            details = self.scrape_book_details(book['url'])
            book.update(details)  # Merge detail fields into the book dict
            time.sleep(1)  # Polite delay between requests

        return books

    def to_dataframe(self, books):
        """Convert list of book dicts to DataFrame and add analysis columns."""
        df = pd.DataFrame(books)

        # Add a price category column using pd.cut() for binning
        df['price_category'] = pd.cut(
            df['price'],
            bins=[0, 20, 40, 60, float('inf')],
            labels=['Budget', 'Mid-range', 'Premium', 'Luxury'],
        )

        return df

    def analyze(self, df):
        """Perform analysis on book data and print a summary report."""
        print("\n" + "=" * 60)
        print("BOOK ANALYSIS REPORT")
        print("=" * 60)

        print(f"\n📊 Total Books: {len(df)}")
        print(f"💰 Average Price: £{df['price'].mean():.2f}")
        print(f"📈 Price Range: £{df['price'].min():.2f} - £{df['price'].max():.2f}")

        print(f"\n⭐ Rating Distribution:")
        print(df['rating'].value_counts().sort_index())

        print(f"\n💵 Books by Price Category:")
        print(df['price_category'].value_counts())

        if 'category' in df.columns:
            print(f"\n📚 Top 5 Categories:")
            print(df['category'].value_counts().head())

        print(f"\n📦 Stock Status:")
        print(df['in_stock'].value_counts())

        print(f"\n🏆 Top 5 Most Expensive Books:")
        top_expensive = df.nlargest(5, 'price')[['title', 'price', 'rating']]
        for idx, row in top_expensive.iterrows():
            print(f"  - {row['title'][:50]}: £{row['price']:.2f} ({row['rating']}★)")


# ── Usage ──────────────────────────────────────────────────────────────────
scraper = BookScraper()

# Scrape 3 pages (20 books per page = 60 books total)
books = scraper.scrape_all_pages(max_pages=3)
print(f"\nTotal books scraped: {len(books)}")

# Convert to DataFrame and run analysis
df = scraper.to_dataframe(books)
scraper.analyze(df)

# Save results to disk
df.to_csv('books_data.csv', index=False)
df.to_excel('books_data.xlsx', index=False)
print("\n💾 Data saved to books_data.csv and books_data.xlsx")

📖 Scraping page 1...
  Found 20 books
📖 Scraping page 2...
  Found 20 books
📖 Scraping page 3...
  Found 20 books

✓ Total books scraped: 60

Total books scraped: 60

BOOK ANALYSIS REPORT

📊 Total Books: 60
💰 Average Price: £35.00
📈 Price Range: £12.84 - £57.31

⭐ Rating Distribution:
rating
1    15
2     8
3    13
4    10
5    14
Name: count, dtype: int64

💵 Books by Price Category:
price_category
Premium      24
Mid-range    22
Budget       14
Luxury        0
Name: count, dtype: int64

📦 Stock Status:
in_stock
True    60
Name: count, dtype: int64

🏆 Top 5 Most Expensive Books:
  - Slow States of Collapse: Poems: £57.31 (3★)
  - Our Band Could Be Your Life: Scenes from the Ameri: £57.25 (3★)
  - The Past Never Ends: £56.50 (4★)
  - The Pioneer Woman Cooks: Dinnertime: Comfort Class: £56.41 (1★)
  - The Secret of Dreadwillow Carse: £56.13 (1★)

💾 Data saved to books_data.csv and books_data.xlsx


### Expected Output:

```
📖 Scraping page 1...
  Found 20 books
📖 Scraping page 2...
  Found 20 books
📖 Scraping page 3...
  Found 20 books

✓ Total books scraped: 60

============================================================
BOOK ANALYSIS REPORT
============================================================

📊 Total Books: 60
💰 Average Price: £35.72
📈 Price Range: £10.00 - £59.99

⭐ Rating Distribution:
1     4
2     8
3    14
4    20
5    14

💵 Books by Price Category:
Mid-range    28
Budget       18
Premium      10
Luxury        4

📦 Stock Status:
True     58
False     2

🏆 Top 5 Most Expensive Books:
  - Book Title Here: £59.99 (5★)
  ...
```

## <span style='color:blue'>3.5 Handling Dynamic Content</span>

**Problem**: Some websites load data with **JavaScript** *after* the initial page load. `BeautifulSoup` only sees the raw HTML — it **cannot execute JavaScript**!

### Solutions:

In [4]:
# ── Option 1: Find the Hidden API (BEST approach) ─────────────────────────
#
# Instead of scraping rendered HTML, find the API the website uses internally!
# Steps:
#   1. Open Chrome DevTools (F12)
#   2. Go to "Network" tab
#   3. Filter by "Fetch/XHR"
#   4. Reload page or interact with the site
#   5. Look for JSON responses
#   6. Copy the URL and use it directly!

# Example: Many sites expose hidden APIs — much faster than scraping!
# import requests

# api_url = 'https://example.com/api/products?page=1'
# response = requests.get(api_url)
# data = response.json()  # Get clean, structured JSON directly — no HTML parsing needed!

# This is MUCH easier and faster than parsing HTML with BeautifulSoup!

In [5]:
!pip install selenium webdriver-manager

In [6]:
# ── Option 2: Use Selenium (If No API is Available) ───────────────────────
#
# Selenium controls a real browser, so JavaScript DOES execute.
# This is more complex and slower, but necessary for heavily JS-driven sites.
# Note: This is beyond the core scope but good to know!

from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from bs4 import BeautifulSoup

# url = 'https://example.com'
url = 'https://www.hackerrank.com/'

# Launch a browser (requires ChromeDriver installed)
driver = webdriver.Chrome()
driver.get(url)

# Wait for a specific element to appear before scraping
# (avoids scraping before JS has finished loading content)
element = WebDriverWait(driver, 10).until(
    EC.presence_of_element_located((By.CLASS_NAME, "js-dynamic-word"))
)
print("Page loaded and dynamic content is present!")
print("Content: ", element.text)

# Now parse the fully-rendered page source with BeautifulSoup
soup = BeautifulSoup(driver.page_source, 'lxml')

driver.quit()  # Always close the browser when done

NoSuchWindowException: Message: no such window: target window already closed
from unknown error: web view not found
  (Session info: chrome=133.0.6943.127)
Stacktrace:
	GetHandleVerifier [0x00007FF68835C6A5+28789]
	(No symbol) [0x00007FF6882C5B20]
	(No symbol) [0x00007FF688158F9A]
	(No symbol) [0x00007FF688131563]
	(No symbol) [0x00007FF6881DEF1E]
	(No symbol) [0x00007FF6881FF0D2]
	(No symbol) [0x00007FF6881D7103]
	(No symbol) [0x00007FF68819FFC0]
	(No symbol) [0x00007FF6881A1273]
	GetHandleVerifier [0x00007FF6886A1AED+3458237]
	GetHandleVerifier [0x00007FF6886B829C+3550316]
	GetHandleVerifier [0x00007FF6886ADB9D+3507565]
	GetHandleVerifier [0x00007FF688422C6A+841274]
	(No symbol) [0x00007FF6882D09EF]
	(No symbol) [0x00007FF6882CCB34]
	(No symbol) [0x00007FF6882CCCD6]
	(No symbol) [0x00007FF6882BC119]
	BaseThreadInitThunk [0x00007FF88936E8D7+23]
	RtlUserThreadStart [0x00007FF88A6CC40C+44]


## <span style='color:blue'>3.6 robots.txt: The Rules</span>

<span style='color:red'>**⚠️ Always check `robots.txt` before scraping!**</span>

The `robots.txt` file at the root of every website tells bots which pages they are and aren't allowed to scrape.

In [ ]:
from urllib.robotparser import RobotFileParser
from urllib.parse import urlparse, urljoin
import requests


def can_scrape(url, user_agent='*'):
    """
    Check if URL can be scraped according to robots.txt

    Args:
        url:        URL to check
        user_agent: Your bot's user agent string

    Returns:
        bool: True if allowed, False otherwise
    """
    try:
        # Build the robots.txt URL from the domain
        parsed = urlparse(url)
        robots_url = f"{parsed.scheme}://{parsed.netloc}/robots.txt"

        # Parse the robots.txt file
        rp = RobotFileParser()
        rp.set_url(robots_url)
        rp.read()

        # Check if our user_agent is allowed to access this URL
        allowed = rp.can_fetch(user_agent, url)

        if allowed:
            print(f"✅ Allowed: {url}")
        else:
            print(f"❌ Blocked: {url}")
            print(f"  Check {robots_url} for rules")

        return allowed

    except Exception as e:
        print(f"⚠️ Could not read robots.txt: {e}")
        print("  Proceed with caution and respect")
        return False


# Test: check if scraping is allowed for these URLs
can_scrape('http://books.toscrape.com/catalogue/page-1.html')
can_scrape('https://www.google.com/search?q=python')

# View the raw robots.txt file directly
response = requests.get('http://books.toscrape.com/robots.txt')
print(response.text)

### robots.txt Example:

```
User-agent: *
Disallow: /admin/
Disallow: /private/
Allow: /public/
Crawl-delay: 1
```

**What it means:**
- `User-agent: *` — applies to **all** bots
- `Disallow: /admin/` — don't scrape anything in `/admin/`
- `Allow: /public/` — `/public/` is OK
- `Crawl-delay: 1` — wait **1 second** between requests

## <span style='color:blue'>3.7 Best Practices & Ethics</span>

### ✅ DO:

**1. Check robots.txt first**
```python
can_scrape(url)
```

**2. Identify yourself**
```python
headers = {
    'User-Agent': 'MyBot/1.0 (+http://mywebsite.com/bot-info)'
}
```

**3. Add delays**
```python
time.sleep(random.uniform(1, 3))  # 1-3 seconds
```

**4. Cache results**
```python
# Don't re-scrape the same page
if url in cache:
    return cache[url]
```

**5. Handle errors gracefully**
```python
try:
    response = requests.get(url)
except Exception as e:
    print(f"Error: {e}")
```

**6. Respect the site**
```python
# Scrape during off-peak hours
# Limit concurrent requests
# Stop if you get blocked
```

---

### ❌ DON'T:

1. **Ignore `robots.txt`**
2. **Scrape personal data without consent**
3. **Overwhelm servers (DDoS)**
4. **Scrape if an API exists**
5. **Ignore copyright**
6. **Scrape login-protected content**

## <span style='color:red'>3.8 Graded Exercise 3: E-Commerce Analysis</span>

**Scenario**: Analyze the book market on [Books to Scrape](http://books.toscrape.com/)

---

### <span style='color:blue'>Task 1: Basic Scraping (15 points)</span>

**1.1** *(8 points)* — Scrape books from the **"Travel"** category
- Navigate to: http://books.toscrape.com/catalogue/category/books/travel_2/index.html
- Extract: `title`, `price`, `rating`, `availability`
- Handle pagination (all pages in category)
- Save as: `task1_travel_books.csv`

In [ ]:
import time
import requests
import pandas as pd
import re
from bs4 import BeautifulSoup
from urllib.parse import urljoin


def parse_rating(article):
    word_to_number = {'One': 1, 'Two': 2, 'Three': 3, 'Four': 4, 'Five': 5}
    rating_element = article.select_one('.star-rating')
    if not rating_element:
        return 0
    classes = rating_element.get('class', [])
    rating_word = classes[1] if len(classes) > 1 else None
    # print(rating_word)
    return word_to_number.get(rating_word, 0)


def parse_price(article):
    price_text = article.select_one('.price_color').get_text(strip=True)
    # print(f"{price_text}")
    clean_text = re.sub(r'[^0-9.]', '', price_text)
    return float(clean_text)


def scrape_travel_books():
    """
    Scrape all travel books.

    Requirements:
    - Handle pagination
    - Add 1 second delay between pages
    - Extract all required fields

    Returns:
        DataFrame with book data
    """
    session = requests.Session()
    session.headers.update({'User-Agent': 'Mozilla/5.0 (Educational) Part3Scraper/1.0'})

    books = []
    page_url = 'http://books.toscrape.com/catalogue/category/books/travel_2/index.html'

    while page_url:
        response = session.get(page_url, timeout=10)
        response.raise_for_status()
        soup = BeautifulSoup(response.text, 'lxml')

        for article in soup.select('article.product_pod'):
            title = article.select_one('h3 a')
            availability = article.select_one('.availability').get_text(' ', strip=True)


            books.append(
                {
                    'title': title.get('title', '').strip(),
                    'price': parse_price(article),
                    'rating': parse_rating(article),
                    'availability': availability,
                }
            )

        next_link = soup.select_one('li.next a')


        page_url = urljoin(page_url, next_link['href']) if next_link else None


        if page_url:
            time.sleep(1)

    return pd.DataFrame(books)


df_travel = scrape_travel_books()
df_travel.to_csv('task1_travel_books.csv', index=False)



print(df_travel.head())

avg_price = df_travel['price'].mean()
five_star_books = (df_travel['rating'] == 5).sum()
in_stock_percentage = df_travel['availability'].str.contains('in stock',case=False, na=False).mean()


analysis_text = (
    f"average price {avg_price:.2f}\n"
    f"number of 5-star books: {five_star_books}\n"
    f"in stock percentage : {in_stock_percentage:.2f}%\n"
)

with open('task1_analysis.txt', 'w', encoding='utf-8') as f:
    f.write(analysis_text)


                                               title  price  rating  \
0                            It's Only the Himalayas  45.17       2   
1  Full Moon over Noahâs Ark: An Odyssey to Mou...  49.43       4   
2  See America: A Celebration of Our National Par...  48.87       3   
3  Vagabonding: An Uncommon Guide to the Art of L...  36.94       2   
4                               Under the Tuscan Sun  37.33       3   

  availability  
0     In stock  
1     In stock  
2     In stock  
3     In stock  
4     In stock  


**1.2** *(7 points)* — Analysis questions:
- What's the average price of travel books?
- How many travel books are rated 5 stars?
- What percentage are in stock?

Save answers in: `task1_analysis.txt`

---

### <span style='color:blue'>Task 2: Multi-Category Comparison (20 points)</span>

**2.1** *(12 points)* — Scrape these categories:
- Fiction
- Mystery
- Historical Fiction
- Science Fiction

For each category:
- Scrape **first 2 pages only**
- Extract: `title`, `price`, `rating`, `category`
- Save combined data as: `task2_categories.csv`

**2.2** *(8 points)* — Comparative analysis:
- Which category has the **highest average rating**?
- Which has the **most expensive books** on average?
- Create a comparison visualization (save as `task2_comparison.png`)

In [ ]:
import time
import requests
import pandas as pd
import matplotlib.pyplot as plt
import re
from bs4 import BeautifulSoup
from urllib.parse import urljoin


class CategoryScraper:
    """
    Scrape multiple categories and compare.
    """

    def __init__(self):
        self.base_url = 'http://books.toscrape.com'
        self.session = requests.Session()

        self.session.headers.update({'User-Agent': 'Mozilla/5.0 (Educational) CategoryScraper/1.0'})
        self.word_to_number = {'One': 1, 'Two': 2, 'Three': 3, 'Four': 4, 'Five': 5}
        self.category_urls = self._get_category_urls()

    def _get_category_urls(self):
        url = f'{self.base_url}/index.html'
        response = self.session.get(url, timeout=10)

        response.raise_for_status()
        soup = BeautifulSoup(response.text, 'lxml')

        mapping = {}
        for link in soup.select('div.side_categories a'):
            name = link.get_text(strip=True)
            if not name:
                continue
            mapping[name.lower()] = urljoin(url, link.get('href'))
        return mapping

    def _parse_price(self, article):
        text = article.select_one('.price_color').get_text(strip=True)
        clean = re.sub(r'[^0-9.]', '', text)
        return float(clean)

    def _parse_rating(self, article):
        classes = article.select_one('.star-rating').get('class', [])
        word = classes[1] if len(classes) > 1 else None
        return self.word_to_number.get(word, 0)


    def scrape_category(self, category_name, max_pages=2):
        page_url = self.category_urls.get(category_name.lower())

        if not page_url:
            raise ValueError(f'not found: {category_name}')

        books = []
        page_num = 1

        while page_url and page_num <= max_pages:

            response = self.session.get(page_url, timeout=10)
            response.raise_for_status()
            soup = BeautifulSoup(response.text, 'lxml')

            for article in soup.select('article.product_pod'):
                title = article.select_one('h3 a').get('title', '').strip()
                books.append(
                    {
                        'title': title,
                        'price': self._parse_price(article),
                        'rating': self._parse_rating(article),
                        'category': category_name,
                    }
                )

            next_link = soup.select_one('li.next a')
            page_url = urljoin(page_url, next_link['href']) if next_link else None
            page_num += 1
            if page_url and page_num <= max_pages:
                time.sleep(1)

        return books

    def scrape_multiple_categories(self, categories):

        all_books = []
        for category in categories:
            all_books.extend(self.scrape_category(category, max_pages=2))
            time.sleep(1)
        return pd.DataFrame(all_books)

    def compare_categories(self, df):

        stats_df = (
            df.groupby('category', as_index=False)
            .agg(
                avg_price=('price', 'mean'),
                avg_rating=('rating', 'mean'),
            )
        )


        highest_rating_category = stats_df.loc[stats_df['avg_rating'].idxmax(), 'category']
        highest_price_category = stats_df.loc[stats_df['avg_price'].idxmax(), 'category']

        fig, axes = plt.subplots(1, 2, figsize=(16, 5))
        axes[0].bar(stats_df['category'], stats_df['avg_rating'])
        axes[0].set_title('Average Rating')
        axes[0].tick_params(axis='x', rotation=27)

        axes[1].bar(stats_df['category'], stats_df['avg_price'])
        axes[1].set_title('Average Price')
        axes[1].tick_params(axis='x', rotation=27)

        fig.tight_layout()
        fig.savefig('task2_comparison.png', dpi=200, bbox_inches='tight')
        plt.close(fig)

        return {
            'highest_average_rating': highest_rating_category,
            'highest_average_price': highest_price_category,
            'stats_table': stats_df,
        }


# Task 2 outputs
categories = ['Fiction', 'Mystery', 'Historical Fiction', 'Science Fiction']
category_scraper = CategoryScraper()
df_categories = category_scraper.scrape_multiple_categories(categories)
df_categories.to_csv('task2_categories.csv', index=False)

comparison = category_scraper.compare_categories(df_categories)

highest rating Historical Fiction
most expensive Fiction
             category  avg_price  avg_rating
0             Fiction  36.455500    3.050000
1  Historical Fiction  33.644231    3.230769
2             Mystery  31.719062    2.937500
3     Science Fiction  33.802500    2.250000


### <span style='color:blue'>Task 3: Advanced Scraping Pipeline (15 points)</span>

**3.1** *(15 points)* — Build a **complete scraping system**

**Requirements:**

1. **Scraper Class** with:
   - Rate limiting (max **10 requests/minute**)
   - Retry logic (**3 attempts** with exponential backoff)
   - Logging (save to `scraper.log`)
   - `robots.txt` checker
   - Progress tracking

2. **Data Validation**:
   - Verify all prices are valid numbers
   - Check all ratings are 1–5
   - Ensure no duplicate books

3. **Error Recovery**:
   - If scraping fails, save progress
   - Resume from last successful page
   - Log all errors with timestamps

4. **Export Options**:
   - CSV with UTF-8 encoding
   - Excel with formatted headers
   - JSON with proper structure

In [ ]:
import os
import re
import time
import json
import logging
from datetime import datetime
from urllib.parse import urljoin, urlparse
from urllib.robotparser import RobotFileParser
import requests
import pandas as pd
from bs4 import BeautifulSoup
from openpyxl.styles import Font, PatternFill


class AdvancedBookScraper:

    def __init__(self, output_dir='.'):
        """
        Initialize scraper with logging and rate limiting.
        """
        self.output_dir = output_dir
        os.makedirs(self.output_dir, exist_ok=True)

        self.logger = logging.getLogger('AdvancedBookScraper')
        self.logger.setLevel(logging.INFO)
        if not self.logger.handlers:
            fh = logging.FileHandler('scraper.log', encoding='utf-8')
            fh.setFormatter(
                logging.Formatter('%(asctime)s - %(levelname)s - %(message)s')
            )
            self.logger.addHandler(fh)

        self.base_url = 'http://books.toscrape.com'
        self.session = requests.Session()
        self.session.headers.update({'User-Agent': 'Mozilla/5.0 (Educational) AdvancedBookScraper/1.0'})

        self.request_timestamps = []
        self.progress_tracker = {}
        self.rating_map = {'One': 1, 'Two': 2, 'Three': 3, 'Four': 4, 'Five': 5}


    def _respect_rate_limit(self):
        now = time.time()
        self.request_timestamps = [t for t in self.request_timestamps if now - t < 60]
        if len(self.request_timestamps) >= 10:
            sleep_for = 60 - (now - self.request_timestamps[0])
            if sleep_for > 0:
                self.logger.info(f'Rate limit reached; sleeping {sleep_for:.2f}s')
                time.sleep(sleep_for)
            now = time.time()
            self.request_timestamps = [t for t in self.request_timestamps if now - t < 60]
        self.request_timestamps.append(time.time())



    def _category_url_map(self):
        soup = self.scrape_with_retry(f'{self.base_url}/index.html')
        if soup is None:
            return {}
        mapping = {}
        for link in soup.select('div.side_categories a'):
            name = link.get_text(strip=True)
            if not name or name.lower() == 'books':
                continue
            mapping[name.lower()] = urljoin(f'{self.base_url}/index.html', link.get('href'))
        return mapping

    def check_robots_txt(self, url):
        parsed = urlparse(url)
        robots_url = f'{parsed.scheme}://{parsed.netloc}/robots.txt'

        try:
            rp = RobotFileParser()
            rp.set_url(robots_url)
            rp.read()
            allowed = rp.can_fetch('*', url)
            self.logger.info(f'robots.txt check for {url}: {allowed}')
            return allowed
        except Exception as e:
            self.logger.error(f'robots.txt check failed: {e}')
            return False


    def scrape_with_retry(self, url, max_attempts=3):
        """
        scape with backoff 1,2,4,...
        """

        for attempt in range(max_attempts):
            try:
                self._respect_rate_limit()
                response = self.session.get(url, timeout=10)
                response.raise_for_status()
                return BeautifulSoup(response.text, 'lxml')
            except Exception as e:
                wait = 2**attempt
                self.logger.error(
                    f'attempt {attempt + 1}/{max_attempts} failed for {url}: {e}'
                )
                if attempt < max_attempts - 1:
                    time.sleep(wait)
        return None


    def _parse_price(self, article):
        text = article.select_one('.price_color').get_text(strip=True)
        clean = re.sub(r'[^0-9.]', '', text)
        return float(clean)

    def _parse_rating(self, article):
        classes = article.select_one('.star-rating').get('class', [])
        word = classes[1] if len(classes) > 1 else None
        return self.rating_map.get(word, 0)

    def validate_book_data(self, book):

        try:
            price_ok = isinstance(book.get('price'), (int, float)) and book['price'] > 0
            rating_ok = isinstance(book.get('rating'), int) and 1 <= book['rating'] <= 5
            title_ok = isinstance(book.get('title'), str) and len(book['title'].strip()) > 0
            return price_ok and rating_ok and title_ok
        except Exception:
            return False

    def save_progress(self, books, state=None, filename='progress.json'):

        payload = {
            'saved_at': datetime.now().isoformat(),
            'books': books,
            'state': state or {},
            'progress_tracker': self.progress_tracker,
        }
        path = os.path.join(self.output_dir, filename)
        with open(path, 'w', encoding='utf-8') as f:
            json.dump(payload, f, ensure_ascii=False, indent=2)

    def load_progress(self, filename='progress.json'):

        path = os.path.join(self.output_dir, filename)
        if not os.path.exists(path):
            return {'books': [], 'state': {}, 'progress_tracker': {}}

        with open(path, 'r', encoding='utf-8') as f:
            data = json.load(f)
        self.progress_tracker = data.get('progress_tracker', {})
        return data

    def export_data(self, books, base_filename='task3_books'):

        df = pd.DataFrame(books)

        csv_path = os.path.join(self.output_dir, f'{base_filename}.csv')
        xlsx_path = os.path.join(self.output_dir, f'{base_filename}.xlsx')
        json_path = os.path.join(self.output_dir, f'{base_filename}.json')

        df.to_csv(csv_path, index=False, encoding='utf-8')

        with pd.ExcelWriter(xlsx_path, engine='openpyxl') as writer:
            df.to_excel(writer, index=False, sheet_name='Books')
            ws = writer.book['Books']

        payload = {
            'generated_at': datetime.now(),
            'total_books': int(len(books)),
            'books': books,
        }
        with open(json_path, 'w', encoding='utf-8') as f:
            json.dump(payload, f, ensure_ascii=False, indent=2)

        return {'csv': csv_path, 'xlsx': xlsx_path, 'json': json_path}

    def run_full_pipeline(self, categories, max_pages_per_category=5):

        test_url = f'{self.base_url}/catalogue/page-1.html'
        if not self.check_robots_txt(test_url):
            raise RuntimeError('Scraping not allowed by robots.txt')

        progress = self.load_progress('progress.json')
        books = progress.get('books', [])
        state = progress.get('state', {})


        category_urls = self._category_url_map()
        if not category_urls:
            raise RuntimeError('cannot fetch category URLs')
        

        for category in categories:
            cat_key = category.lower()
            if cat_key not in category_urls:
                self.logger.error(f'skipping missing category: {category}')
                continue

            category_state = state.get(category, {})
            page_url = category_state.get('next_url') or category_urls[cat_key]
            page_num = int(category_state.get('next_page', 1))

            self.logger.info(f'start category {category} from page {page_num}')

            while page_url and page_num <= max_pages_per_category:
                page_id = f'{category}|{page_num}'
                soup = self.scrape_with_retry(page_url, max_attempts=3)
                if soup is None:
                    self.progress_tracker[page_id] = 'failed'
                    self.save_progress(books, state, 'progress.json')
                    self.logger.error(f'Failed page {page_id}; progress saved')
                    break



                for article in soup.select('article.product_pod'):
                    book = {
                        'title': article.select_one('h3 a').get('title', '').strip(),
                        'price': self._parse_price(article),
                        'rating': self._parse_rating(article),
                        'availability': article.select_one('.availability').get_text(' ', strip=True),
                        'category': category,
                    }



                    if self.validate_book_data(book):
                        books.append(book)
                    else:
                        self.logger.error(f'invalid data skipped: {book}')

                next_link = soup.select_one('li.next a')
                next_url = urljoin(page_url, next_link['href']) if next_link else None



                self.progress_tracker[page_id] = 'completed'
                state[category] = {
                    'next_page': page_num + 1,
                    'next_url': next_url,
                }
                self.save_progress(books, state, 'progress.json')
                self.logger.info(
                    f'completed {page_id}; added {len(books)} books; total {len(books)}'
                )

                page_url = next_url
                page_num += 1

        exports = self.export_data(books, base_filename='task3_books')

        summary_path = os.path.join(self.output_dir, 'task3_summary.txt')
        with open(summary_path, 'w', encoding='utf-8') as f:
            f.write(f'total number of books: {len(books)}\n')
            f.write('books per category:\n')
            if books:
                counts = pd.DataFrame(books)['category'].value_counts()
                for cat, cnt in counts.items():
                    f.write(f'- {cat}: {cnt}\n')



        self.logger.info('pipeline completed successfully')
        return {
            'total_books': len(books),
            'exports': exports,
            'summary': summary_path,
            'progress_file': os.path.join(self.output_dir, 'progress.json'),
        }
    


scraper = AdvancedBookScraper()
result = scraper.run_full_pipeline(
    categories=['Mystery', 'Science Fiction', 'Fantasy'], max_pages_per_category=3
)
print(result)

{'total_books': 95, 'exports': {'csv': '.\\task3_books.csv', 'xlsx': '.\\task3_books.xlsx', 'json': '.\\task3_books.json'}, 'summary': '.\\task3_summary.txt', 'progress_file': '.\\progress.json'}


---

### <span style='color:red'>📋 Submission Requirements</span>

**Files to submit:**

| # | File | Description |
|---|------|-------------|
| 1 | `book_scraper.py` | All code |
| 2 | `task1_travel_books.csv` | Travel books data |
| 3 | `task1_analysis.txt` | Task 1 analysis answers |
| 4 | `task2_categories.csv` | Multi-category data |
| 5 | `task2_comparison.png` | Comparison visualization |
| 6 | `task3_books.csv` | Advanced scraper CSV output |
| 7 | `task3_books.xlsx` | Advanced scraper Excel output |
| 8 | `task3_books.json` | Advanced scraper JSON output |
| 9 | `scraper.log` | Log file |
| 10 | `README.md` | Documentation |

**`README.md` should include:**
- How to run your code
- Dependencies needed
- Any issues encountered
- Key findings from analysis

---

### <span style='color:blue'>📊 Grading Rubric</span>

| Category | Weight | Criteria |
|---|---|---|
| **Functionality** | 40% | Correct data extraction, proper pagination, error handling |
| **Code Quality** | 30% | Clean & readable code, proper comments, logging implementation |
| **Data Quality** | 20% | Validation implemented, no duplicates, correct formatting |
| **Analysis** | 10% | Insights from data, visualization quality |